In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
"""
Phase 5: RAG Agent
Takes a user question, retrieves relevant chunks from the fixed-size Vector
Search index (selected in Phase 6 evaluation), constructs a grounded prompt,
and generates an answer using Databricks' hosted GPT OSS 120B model.
"""

import mlflow.deployments
from openai import OpenAI
from databricks.vector_search.client import VectorSearchClient

# --- Setup: clients and index ---
embed_client = mlflow.deployments.get_deploy_client("databricks")
vsc = VectorSearchClient()

fixed_index = vsc.get_index(
    endpoint_name="rag_pipeline_endpoint",
    index_name="rag_pipeline.main.fixed_size_index"
)

DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
llm_client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url="https://7474646723482977.ai-gateway.cloud.databricks.com/mlflow/v1"
)



/home/spark-79b3bd0d-455f-4f0f-ab05-c1/.ipykernel/73/command-8865504264522844-3620170317:10: DeprecationWarning: databricks-vectorsearch is deprecated and has been renamed to databricks-ai-search. Imports under 'databricks.vector_search.*' will continue to work as a thin re-export of 'databricks.ai_search.*', but new code should switch to 'pip install databricks-ai-search' and 'from databricks.ai_search.* import ...'.
  from databricks.vector_search.client import VectorSearchClient


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


In [0]:
# Embed the user's query 
def embed_query(text):
    response = embed_client.predict(
        endpoint="databricks-bge-large-en",
        inputs={"input": [text]}
    )
    return response["data"][0]["embedding"]


# Retrieve top-k relevant chunks
def retrieve_chunks(query, num_results=5, min_score=0.5):
    query_vector = embed_query(query)
    results = fixed_index.similarity_search(
        query_vector=query_vector,
        columns=["chunk_id", "company", "fiscal_year", "chunk_text"],
        num_results=num_results
    )
    # Vector Search doesn't return score in data_array by default unless you request it —
    # add "score" implicitly via the columns/result structure, or check via manual cosine sim if needed
    return results["result"]["data_array"]


# Build a grounded prompt from retrieved chunks 
def build_prompt(query_text, chunks):
    context_blocks = []
    for chunk in chunks:
        chunk_id, company, fiscal_year, chunk_text = chunk[0], chunk[1], chunk[2], chunk[3]
        context_blocks.append(
            f"[Source: {company} {fiscal_year} 10-K]\n{chunk_text}"
        )
    context = "\n\n---\n\n".join(context_blocks)

    prompt = f"""You are a financial analyst assistant. Answer the question using ONLY the context below, which is drawn from SEC 10-K filings. If the context doesn't contain enough information to answer, say so clearly rather than guessing. Cite which company/year each fact comes from when relevant.

CONTEXT:
{context}

QUESTION:
{query_text}

ANSWER:"""
    return prompt


In [0]:
#  Call the LLM and extract clean text
def generate_answer(prompt, max_tokens=800):
    response = llm_client.chat.completions.create(
        model="databricks-gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    content = response.choices[0].message.content
    if isinstance(content, list):
        answer = next((block["text"] for block in content if block["type"] == "text"), None)
    else:
        answer = content
    return answer


# Full RAG pipeline
def rag_query(query_text, num_results=5, verbose=True):
    chunks = retrieve_chunks(query_text, num_results=num_results)
    prompt = build_prompt(query_text, chunks)
    answer = generate_answer(prompt)

    if verbose:
        print("QUESTION:", query_text)
        print("\nRETRIEVED SOURCES:")
        for c in chunks:
            print(f"  - {c[1]} {c[2]} (chunk_id={c[0]})")
        print("\nANSWER:\n", answer)

    return answer

In [0]:
import json
import re

tools = [
    {
        "type": "function",
        "function": {
            "name": "search_10k_filings",
            "description": "Search SEC 10-K filings from NVIDIA, Apple, Microsoft, Meta, and Alphabet (2023-2025) for relevant passages. Use this whenever you need specific facts, figures, or disclosures from these filings.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "A specific search query describing what information you need."},
                    "num_results": {"type": "integer", "description": "Number of chunks to retrieve (default 5).", "default": 5}
                },
                "required": ["query"]
            }
        }
    }
]

def run_agent(user_question, max_turns=6, verbose=True):
    messages = [
        {"role": "system", "content": "You are a financial analyst assistant with access to a search tool over SEC 10-K filings from NVIDIA, Apple, Microsoft, Meta, and Alphabet (2023-2025). Use the search tool to find grounded facts before answering. If your first search doesn't return relevant results, try reformulating your query and searching again before giving up. Always cite the company and fiscal year for any fact you state. If you cannot find relevant information after searching, say so clearly. CRITICAL: Before answering, check whether the retrieved passages actually discuss the topic asked about. If the passages are about unrelated topics (e.g., stock price data, auditor procedures, boilerplate legal text) rather than the specific subject of the question, you MUST say the search did not find relevant information — do NOT fill in plausible-sounding details from general knowledge, even if you believe they are likely true. Every specific fact, figure, or claim must be traceable to an exact phrase in the retrieved passages. If you cannot point to the exact source text for a claim, do not include it."},
        {"role": "user", "content": user_question}
    ]
    all_retrieved_chunks = []  # track everything retrieved across the whole run

    for turn in range(max_turns):
        response = llm_client.chat.completions.create(
            model="databricks-gpt-oss-120b",
            messages=messages,
            tools=tools,
            max_tokens=2000,
            extra_body={"reasoning_effort": "low"}
        )

        msg = response.choices[0].message

        # Check if the model wants to call a tool
        if msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content, "tool_calls": msg.tool_calls})

            for tool_call in msg.tool_calls:
                args = json.loads(tool_call.function.arguments)
                query = args["query"]
                num_results = args.get("num_results", 5)

                if verbose:
                    print(f"[Turn {turn+1}] Agent searching: '{query}' (top {num_results})")

                chunks = retrieve_chunks(query, num_results=num_results)
                all_retrieved_chunks.extend(chunks)
                tool_result = "\n\n".join([
                    f"[{c[1]} {c[2]}]: {c[3]}" for c in chunks
                ])

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_result
                })
            
            print("\n=== ALL CHUNKS RETRIEVED THIS RUN ===")
            for c in all_retrieved_chunks:
                print(f"[{c[1]} {c[2]}] {c[3][:200]}...")
        

        else:
            content = msg.content

            # Fallback: check if a tool call got "narrated" in reasoning instead of properly emitted
            if isinstance(content, list):
                reasoning_text = ""
                for block in content:
                    if block["type"] == "reasoning":
                        for s in block.get("summary", []):
                            reasoning_text += s.get("text", "")

                json_match = re.search(r'\{\s*"query".*?\}', reasoning_text, re.DOTALL)
                if json_match:
                    try:
                        fake_call = json.loads(json_match.group(0))
                        query = fake_call["query"]
                        num_results = fake_call.get("num_results", 5)
                        if verbose:
                            print(f"[Turn {turn+1}] Recovered narrated tool call: '{query}' (top {num_results})")
                        chunks = retrieve_chunks(query, num_results=num_results)
                        tool_result = "\n\n".join([f"[{c[1]} {c[2]}]: {c[3]}" for c in chunks])
                        messages.append({"role": "assistant", "content": content})
                        messages.append({"role": "user", "content": f"Search results for '{query}':\n\n{tool_result}"})
                        continue  # go to next turn instead of returning
                    except (json.JSONDecodeError, KeyError):
                        pass

            # Normal text extraction fallback
            answer = next((b["text"] for b in content if b["type"] == "text"), None) if isinstance(content, list) else content
            if answer is None:
                answer = "I wasn't able to find enough information in the filings to answer this question confidently."
            if verbose:
                print(f"\nFINAL ANSWER:\n{answer}")
            return answer

    # If we've used all turns without a final answer, force synthesis
    final_response = llm_client.chat.completions.create(
    model="databricks-gpt-oss-120b",
    messages=messages + [{"role": "user", "content": "Based ONLY on the specific search results retrieved above, provide your answer now. Do not include any fact, figure, dollar amount, percentage, or specific detail that does not appear verbatim or near-verbatim in the retrieved passages. If the retrieved passages are insufficient to support a detailed answer, give a brief, honest answer stating what is missing rather than filling gaps with plausible-sounding details."}],
    max_tokens=1000,
    extra_body={"reasoning_effort": "low"}
    )
    content = final_response.choices[0].message.content
    answer = next((b["text"] for b in content if b["type"] == "text"), None) if isinstance(content, list) else content
    if answer is None:
        answer = "The agent searched multiple times but could not find sufficient information to answer this question confidently."
    if verbose:
        print(f"\nFINAL ANSWER (forced synthesis):\n{answer}")
    return answer

In [0]:
# cross-year query
run_agent("How did Meta's regulatory risk disclosures change between 2023 and 2025?")

[Turn 1] Agent searching: 'Meta regulatory risk disclosures 2023 Form 10-K' (top 5)
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

=== ALL CHUNKS RETRIEVED THIS RUN ===
[META 2024] o any filing
under the Securities Act of 1933, as amended, or the Exchange Act.
Item 16. Form 10-K Summary
None.
127
7/28/26, 9:32 AM meta-20241231
https://www.sec.gov/Archives/edgar/data/1326801/0001...
[META 2024] n the closing price reported for such date on the Nasdaq Global Select Market. On January 24,
2025, the registrant had 2,189,898,148 shares of Class A common stock and 343,761,117 shares of Class B co...
[AAPL 2024] Report on Form 10-K where indicated. The Registrant’s deﬁnitive proxy statement will be ﬁled with the U.S. Securities and Exchange Commission
within 120 days after the end of the ﬁscal year to which t...
[AAPL 2023] orporat

'I’m sorry, but after searching the 2023‑2025 Form\u202f10‑K filings for Meta Platforms, Inc., I was unable to locate passages that detail how Meta’s regulatory‑risk disclosures specifically changed from the 2023 filing to the 2025 filing. The sections that discuss “Risk Factors” (Item\u202f1A) in those reports contain many generic statements about regulatory risk, but the filings do not provide a clear, side‑by‑side description of what was added, removed, or otherwise altered between those years.\n\nIf you need a precise comparison of the regulatory‑risk language across those years, you may need to review the full Item\u202f1A “Risk Factors” sections of the 2023 and 2025 10‑K reports directly.'

In [0]:
# Basic Test Query 
rag_query("What are NVIDIA's supply chain risk factors?")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
QUESTION: What are NVIDIA's supply chain risk factors?

RETRIEVED SOURCES:
  - NVDA 2024 (chunk_id=5711.0)
  - NVDA 2024 (chunk_id=5743.0)
  - NVDA 2024 (chunk_id=5781.0)
  - NVDA 2023 (chunk_id=5372.0)
  - NVDA 2024 (chunk_id=5754.0)

ANSWER:
 **NVIDIA’s supply‑chain risk factors (as disclosed in its recent Form 10‑K filings)**  

| Risk factor | Description | Source |
|-------------|-------------|--------|
| **Export controls** – Restrictions on GPUs, semiconductors and high‑speed networking components can limit resale, repair, or transfer of NVIDIA products, disrupt the supply chain, and reduce demand. Export controls may affect downstream users, CSPs, and customers worldwide (including outside China) and could give competitors an advantage. | • “Export controls targeting GPUs

'**NVIDIA’s supply‑chain risk factors (as disclosed in its recent Form\u202f10‑K filings)**  \n\n| Risk factor | Description | Source |\n|-------------|-------------|--------|\n| **Export controls** – Restrictions on GPUs, semiconductors and high‑speed networking components can limit resale, repair, or transfer of NVIDIA products, disrupt the supply chain, and reduce demand. Export controls may affect downstream users, CSPs, and customers worldwide (including outside China) and could give competitors an advantage. | • “Export controls targeting GPUs and semiconductors associated with AI have subjected… downstream users… to additional restrictions… Export controls could disrupt our supply chain and distribution channels…” | NVDA\u202f2024\u202f10‑K |\n| **Geographic concentration of warehousing** – A substantial portion of NVIDIA’s products is warehoused and distributed from Hong\u202fKong; export controls that affect shipments from this hub could materially disrupt the supply and distr

In [0]:
# cross-company query
rag_query("Compare how these companies describe risks from AI competition.")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
QUESTION: Compare how these companies describe risks from AI competition.

RETRIEVED SOURCES:
  - MSFT 2023 (chunk_id=4035.0)
  - MSFT 2024 (chunk_id=4470.0)
  - META 2025 (chunk_id=3357.0)
  - NVDA 2025 (chunk_id=6229.0)
  - META 2025 (chunk_id=3355.0)

ANSWER:
 **How the companies frame AI‑competition risk (based on the excerpts provided)**  

| Company | Year | What the filing says about competition‑related AI risk | How it is described |
|---------|------|--------------------------------------------------------|---------------------|
| **Microsoft** | 2023 10‑K | The passage talks about AI‑related reputational, legal, and performance risks (e.g., flawed algorithms, biased data, harmful content) but **does not mention competition** as a specific risk. | No explicit competition

'**How the companies frame AI‑competition risk (based on the excerpts provided)**  \n\n| Company | Year | What the filing says about competition‑related AI risk | How it is described |\n|---------|------|--------------------------------------------------------|---------------------|\n| **Microsoft** | 2023\u202f10‑K | The passage talks about AI‑related reputational, legal, and performance risks (e.g., flawed algorithms, biased data, harmful content) but **does not mention competition** as a specific risk. | No explicit competition risk is disclosed in the excerpt. |\n| Microsoft | 2024\u202f10‑K | Similar to 2023, the focus is on potential liability, regulatory action, brand/reputational harm, and content‑related issues. Competition is **not referenced**. | No explicit competition risk is disclosed in the excerpt. |\n| **Meta** | 2025\u202f10‑K (second excerpt) | “…competition from AI features and technologies that may be similar or superior to our technologies or more cost‑effective t

In [0]:
# stress test for grounding, ask something the corpus likely doesn't cover well
rag_query("What is NVIDIA's plan for entering the automotive insurance market?")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
QUESTION: What is NVIDIA's plan for entering the automotive insurance market?

RETRIEVED SOURCES:
  - NVDA 2025 (chunk_id=6074.0)
  - NVDA 2023 (chunk_id=5237.0)
  - NVDA 2025 (chunk_id=6081.0)
  - NVDA 2023 (chunk_id=5231.0)
  - NVDA 2025 (chunk_id=6075.0)

ANSWER:
 The provided excerpts discuss NVIDIA’s DRIVE platform, its AI‑based hardware and software solutions for autonomous vehicles and EVs, and the company’s broader accelerated‑computing strategy, but they contain no reference to a plan to enter the automotive insurance market. Therefore, based on the given context, there is insufficient information to answer the question.


'The provided excerpts discuss NVIDIA’s\u202fDRIVE\u202fplatform, its AI‑based hardware and software solutions for autonomous vehicles and EVs, and the company’s broader accelerated‑computing strategy, but they contain no reference to a plan to enter the automotive insurance market. Therefore, based on the given context, there is insufficient information to answer the question.'